# Mapeamento eleitoral de São João del-Rei – Eleição presidencial 2026


Dados de entrada (gerados pelo `recorte_sjdr.py` a partir dos arquivos oficiais do TSE):
- `dados_sjdr/votacao_presidente_sjdr.csv`: votos para presidente por seção, 1º e 2º turnos
- `dados_sjdr/locais_votacao_sjdr.csv`: locais de votação de cada seção, com latitude e longitude

## 0. Bibliotecas e leitura dos dados

Lemos tudo como texto (`dtype=str`) para não perder nada na leitura. Depois convertemos só as colunas numéricas que vamos usar.

In [42]:
import pandas as pd

votos = pd.read_csv("dados_sjdr/votacao_presidente_sjdr.csv", sep=";", dtype=str)
locais = pd.read_csv("dados_sjdr/locais_votacao_sjdr.csv", sep=";", dtype=str)

votos["QT_VOTOS"] = votos["QT_VOTOS"].astype(int)

print("votos:", votos.shape, "| locais:", locais.shape)

votos: (2706, 26) | locais: (482, 41)


## 2º turno

### Passo 1 – Filtrar o 2º turno

Os dois arquivos trazem os dois turnos. Ficamos só com as linhas do 2º.

In [43]:
votos_2t = votos[votos["NR_TURNO"] == "2"].copy()
locais_2t = locais[locais["NR_TURNO"] == "2"].copy()

print("linhas de votos:", len(votos_2t), "| linhas de locais:", len(locais_2t))
votos_2t.groupby("NM_VOTAVEL")["QT_VOTOS"].sum()

linhas de votos: 855 | linhas de locais: 241


NM_VOTAVEL
FLAVIO NANTES BOLSONARO      27777
LUIZ INÁCIO LULA DA SILVA    27644
VOTO BRANCO                    784
VOTO NULO                     1278
Name: QT_VOTOS, dtype: int64

### Passo 2 – Pivotar: uma linha por seção

Hoje cada linha é "seção × candidato" (4 linhas por seção: Flávio, Lula, branco e nulo).
O `pivot_table` transforma isso em **uma linha por seção**, com uma coluna para cada opção de voto.

In [44]:
por_secao = votos_2t.pivot_table(
    index=["NR_ZONA", "NR_SECAO"],   # o que vira linha
    columns="NM_VOTAVEL",            # o que vira coluna
    values="QT_VOTOS",               # o que vai dentro das células
    aggfunc="sum",
    fill_value=0,                    # se um candidato teve 0 votos na seção, a linha não existe no TSE
).reset_index()
por_secao.columns.name = None

# Nomes curtos para as colunas
por_secao = por_secao.rename(columns={
    "FLAVIO NANTES BOLSONARO": "FLAVIO",
    "LUIZ INÁCIO LULA DA SILVA": "LULA",
    "VOTO BRANCO": "BRANCO",
    "VOTO NULO": "NULO",
})

print("seções com votos:", len(por_secao))
por_secao.head()

seções com votos: 218


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO
0,328,100,108,171,6,6
1,328,102,93,151,4,6
2,328,103,118,138,5,4
3,328,104,116,144,4,6
4,328,105,119,153,3,12


### Passo 3 – Cruzar votos com locais (chave: `NR_ZONA` + `NR_SECAO`)

A tabela de votos não tem latitude/longitude e a de locais não tem votos. Juntamos as duas pela chave **zona + seção**.

**Padronização das chaves:** as duas colunas precisam estar no mesmo formato nas duas tabelas
(texto sem espaços). Se uma tivesse `"69"` e a outra `69` ou `"069"`, o `merge` não encontraria o par.

**Seções agregadas:** no arquivo de locais há seções do tipo *Agregada*. Os eleitores delas votam na urna
de outra seção (a *Principal*), então elas não têm votos próprios: os votos já estão somados na principal.
Usamos só as principais para não contar nada duas vezes.

In [45]:
# Padronizando as chaves: texto, sem espaços extras
for tabela in (por_secao, locais_2t):
    for coluna in ("NR_ZONA", "NR_SECAO"):
        tabela[coluna] = tabela[coluna].str.strip()

print(locais_2t["DS_TIPO_SECAO_AGREGADA"].value_counts())

locais_principais = locais_2t[locais_2t["DS_TIPO_SECAO_AGREGADA"] == "Principal"]

colunas_local = ["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO",
                 "DS_ENDERECO", "NM_BAIRRO", "NR_LATITUDE", "NR_LONGITUDE"]

secoes = por_secao.merge(
    locais_principais[colunas_local],
    on=["NR_ZONA", "NR_SECAO"],
    how="left",                 # mantém todas as seções com voto
    validate="one_to_one",      # dá erro se alguma seção aparecer repetida
)

# Conferência: nenhuma seção com voto pode ficar sem localização
print("seções sem lat/long:", secoes["NR_LATITUDE"].isna().sum())
secoes.head()

DS_TIPO_SECAO_AGREGADA
Principal    218
Agregada      23
Name: count, dtype: int64


seções sem lat/long: 0


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NM_BAIRRO,NR_LATITUDE,NR_LONGITUDE
0,328,100,108,171,6,6,1104,ESCOLA MUNICIPAL DR. KLEBER VASQUES FILGUEIRAS,"RUA RUTH NEDER ISSA, N 50",TEJUCO,"-21,1416078","-44,276157"
1,328,102,93,151,4,6,1171,CENTRO SOCIAL SAO FRANCISCO,"RUA ESCRIVAO FARNESE SILVA, S/N",BONFIM,"-21,1435736","-44,2583195"
2,328,103,118,138,5,4,1171,CENTRO SOCIAL SAO FRANCISCO,"RUA ESCRIVAO FARNESE SILVA, S/N",BONFIM,"-21,1435736","-44,2583195"
3,328,104,116,144,4,6,1180,ESCOLA ESTADUAL INACIO PASSOS,"PRACA DR GUILHERME MILWARD, S/N",BONFIM,"-21,140479","-44,25804"
4,328,105,119,153,3,12,1180,ESCOLA ESTADUAL INACIO PASSOS,"PRACA DR GUILHERME MILWARD, S/N",BONFIM,"-21,140479","-44,25804"


O TSE escreve as coordenadas com vírgula decimal (`-21,1374636`). Convertemos para número com ponto.

In [46]:
for coluna in ("NR_LATITUDE", "NR_LONGITUDE"):
    secoes[coluna] = secoes[coluna].str.replace(",", ".").astype(float)

secoes[["NR_LATITUDE", "NR_LONGITUDE"]].describe()

,NR_LATITUDE,NR_LONGITUDE
count,218.000000,218.000000
mean,-21.144832,-44.263048
std,0.048779,0.051266
min,-21.374600,-44.542755
25%,-21.140593,-44.265646
50%,-21.134055,-44.254490
75%,-21.126847,-44.240083
max,-21.084874,-44.072441


### Passo 4 – Agrupar por ponto no mapa (coordenada)

Várias seções funcionam no mesmo prédio e têm **a mesma coordenada**. Se cada seção virasse um círculo,
eles ficariam empilhados no mapa.

Primeiro tentamos agrupar pelo **código do local** (`NR_LOCAL_VOTACAO`), mas o mapa ainda mostrou círculos
um dentro do outro: alguns prédios aparecem no TSE com **dois códigos de local diferentes e a mesma coordenada**
(ex.: UFSJ – Campus Dom Bosco e a "Seção Especial" do mesmo campus; E.E. Tomé Portes del Rei; E.M. Maria Teresa;
E.M. Bom Pastor). Por isso agrupamos pela **coordenada**: um círculo por ponto do mapa, somando todos os votos
daquele lugar. Os nomes e as seções de cada ponto ficam guardados para o tooltip.

In [47]:
def juntar_textos(valores):
    """Junta textos diferentes sem repetir (ex.: dois nomes do mesmo prédio)."""
    unicos = [v for v in dict.fromkeys(valores) if isinstance(v, str) and v.strip()]
    return " / ".join(unicos)


por_local = secoes.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(
    NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", juntar_textos),
    NR_LOCAL_VOTACAO=("NR_LOCAL_VOTACAO", juntar_textos),
    NM_BAIRRO=("NM_BAIRRO", juntar_textos),
    DS_ENDERECO=("DS_ENDERECO", "first"),
    FLAVIO=("FLAVIO", "sum"),
    LULA=("LULA", "sum"),
    BRANCO=("BRANCO", "sum"),
    NULO=("NULO", "sum"),
    QT_SECOES=("NR_SECAO", "count"),
    SECOES=("NR_SECAO", lambda s: ", ".join(sorted(s, key=int))),
)

print("pontos no mapa:", len(por_local))
print("códigos de local diferentes:", secoes["NR_LOCAL_VOTACAO"].nunique())

# Os pontos que juntaram mais de um código de local (vale citar no relatório)
por_local[por_local["NR_LOCAL_VOTACAO"].str.contains("/")][["NM_LOCAL_VOTACAO", "NR_LOCAL_VOTACAO", "SECOES"]]

pontos no mapa: 62
códigos de local diferentes: 66


,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,SECOES
30,ESCOLA MUNICIPAL MARIA TERESA,1058 / 1066,"40, 41, 67, 68, 69"
32,ESCOLA MUNICIPAL BOM PASTOR (ANTIGA ESCOLA PIP...,1813 / 1830,"307, 310, 311, 312, 313"
40,ESCOLA ESTADUAL TOMÉ PORTES DEL REI,1775 / 1821,"284, 285, 286, 287, 288, 289, 290, 291, 308, 309"
50,UFSJ - CAMPUS DOM BOSCO / SEÇÃO ESPECIAL-UFSJ ...,1252 / 1686,"125, 126, 127, 133, 134, 149, 345"


### Passo 5 – Percentuais sobre os votos válidos

Votos válidos = votos nos candidatos, sem brancos e nulos. É o critério oficial do TSE para definir o vencedor,
e com ele `% Flávio + % Lula = 100%`, então **50% é exatamente o empate**: o ponto neutro da paleta divergente.

In [48]:
por_local["VALIDOS"] = por_local["FLAVIO"] + por_local["LULA"]
por_local["TOTAL"] = por_local["VALIDOS"] + por_local["BRANCO"] + por_local["NULO"]
por_local["PCT_FLAVIO"] = 100 * por_local["FLAVIO"] / por_local["VALIDOS"]
por_local["PCT_LULA"] = 100 * por_local["LULA"] / por_local["VALIDOS"]

# Conferência: os totais da cidade precisam bater com o Passo 1
print(por_local[["FLAVIO", "LULA", "BRANCO", "NULO"]].sum())

por_local.sort_values("PCT_LULA").head(10)

FLAVIO    27777
LULA      27644
BRANCO      784
NULO       1278
dtype: int64


,NR_LATITUDE,NR_LONGITUDE,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,NM_BAIRRO,DS_ENDERECO,FLAVIO,LULA,BRANCO,NULO,QT_SECOES,SECOES,VALIDOS,TOTAL,PCT_FLAVIO,PCT_LULA
0,-21.374600,-44.413545,CAPELA DA C0MUNIDADE DO ZUEIRA,1589,POVOADO DO ZUEIRA,ZONA RURAL,75,8,1,0,1,186,83,84,90.361446,9.638554
5,-21.331336,-44.493679,SALÃO COMUNITÁRIO DO ENGENHO DE SERRA,1481,POVOADO ENGENHO DE SERRA,ZONA RURAL,63,21,2,2,1,172,84,88,75.000000,25.000000
9,-21.258726,-44.201051,CASA PAROQUIAL CAPELA MORRO GRANDE,1724,ZONA RURAL,POVOADO DO MORRO GRANDE,61,27,4,2,1,225,88,94,69.318182,30.681818
11,-21.237010,-44.374351,SALÃO DA IGREJA DE SÃO JANUÁRIO,1473,POVOADO DE JANUÁRIO,"RUA LUIZ SAVIO DE MOURA, 100 - ZONA RURAL",114,69,4,7,1,171,183,194,62.295082,37.704918
4,-21.331740,-44.443482,SALÃO PAROQUIAL DE CRUZEIRO DA BARRA,1538,POVOADO DE CRUZEIRO DA BARRA,ZONA RURAL,30,20,1,3,1,180,50,54,60.000000,40.000000
61,-21.084874,-44.233308,GRUPO ESCOLAR MUNICIPAL PARADA DO GIAROLA,1759,COLÔNIA DO GIAROLA,"R. AMERICO DEODORO BRIGHENTI, S/N",261,177,3,7,2,"230, 231",438,448,59.589041,40.410959
41,-21.128380,-44.234120,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,1864,MATOSINHOS,"AV. SETE DE SETEMBRO, S/N",1373,951,35,57,8,"319, 320, 321, 322, 323, 324, 325, 326",2324,2416,59.079174,40.920826
30,-21.137464,-44.258335,ESCOLA MUNICIPAL MARIA TERESA,1058 / 1066,CENTRO,"PRACA DOS EXPEDICIONARIOS, S/N",678,476,12,25,5,"40, 41, 67, 68, 69",1154,1191,58.752166,41.247834
46,-21.126847,-44.240350,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,1716,MATOSINHOS,"PRAÇA BOM JESUS DE MATOZINHOS, S/N",1233,871,27,43,8,"217, 218, 219, 220, 221, 222, 223, 224",2104,2174,58.602662,41.397338
49,-21.125265,-44.242422,COLEGIO ATUS (ANTIGA ESCOLA DEZ),1791,JARDIM PAULO CAMPOS,RUA JOSE DE RESENDE CAMPOS 479,767,548,16,30,5,"297, 298, 299, 300, 301",1315,1361,58.326996,41.673004


### Passo 6 – Mapa do 2º turno

**Decisões já tomadas:**
- Um círculo por ponto do mapa, com **área proporcional ao total de votos**.
- **Paleta divergente vermelho ↔ azul** (ColorBrewer `RdBu`): vermelho = Lula (13, PT), azul = Flávio (22, PL).
- A cor representa o **% de Lula sobre os votos válidos**. O branco/neutro fica exatamente em **50% (empate)**.

**Por que não usar a escala automática do Folium:** o padrão divide o intervalo entre o menor (9,6%) e o maior (85,2%)
valor dos dados, o que colocaria o meio da escala em 47,4%. Locais onde Flávio venceu por pouco
ficariam com a cor neutra ou até vermelha. Por isso fixamos os limites de forma simétrica em torno de 50%.

**Interatividade:** os pontos viram um **GeoDataFrame** (geopandas) e são desenhados com
`folium.GeoJson` usando `folium.CircleMarker` como marcador. Isso permite usar o **`GeoJsonTooltip`**.
O tooltip é um **cartão em HTML/CSS** montado no pandas, com: nome do local, zona e seções, bairro,
candidato líder (na cor do partido), uma barra com a proporção Lula × Flávio e o total de votos.

**Classes:** testamos duas opções simétricas em torno de 50%:
- **Opção A:** 6 classes (mesmo número do padrão do Folium), com as pontas abertas: até 30 | 30–40 | 40–50 ‖ 50–60 | 60–70 | 70 ou mais
- **Opção B:** 8 classes de 10 pontos, de 10% a 90%

**Escolhemos a A.** Na B, a classe 80–90% ficou vazia (uma cor na legenda que não aparece no mapa) e as classes
extras só separavam 2 povoados pequenos na ponta azul, diferença quase imperceptível no mapa. Os dados estão
concentrados no meio: 86% dos votos estão em locais entre 40% e 60%. A tabela abaixo mostra quantos locais caem
em cada classe nas duas opções.

**Legenda:** a legenda padrão do Folium (branca) desenha cada faixa com largura proporcional ao valor, então a
faixa "até 30" ocupava um terço da barra e as faixas do meio ficavam espremidas. Fizemos uma legenda em HTML
com as 6 faixas do mesmo tamanho.

**Empates:** dois locais tiveram exatamente o mesmo número de votos para os dois candidatos (Centro Educacional
Frei Seráfico e Unidade Regional de Saúde). Com 50%, eles cairiam na classe 50–60 e pareceriam vitória do Lula.
Por isso recebem uma cor própria, **branco** (o centro neutro da paleta divergente), com um item "Empate" na legenda.
Testamos também cinza, mas ele se confundia com o mapa de fundo, que é cinza; a borda cinza dos círculos
mantém os brancos visíveis.

In [49]:
import numpy as np
import folium
import branca.colormap as cm
import geopandas as gpd

# Cores do ColorBrewer RdBu, do azul (Lula baixo = Flávio alto) para o vermelho (Lula alto)
RDBU_6 = ["#2166ac", "#67a9cf", "#d1e5f0", "#fddbc7", "#ef8a62", "#b2182b"]
RDBU_8 = ["#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b"]
VERMELHO_LULA, AZUL_FLAVIO = "#b2182b", "#2166ac"   # as pontas da paleta, usadas também no tooltip

# Bordas das classes (% de Lula nos votos válidos). Repare que 50 é sempre uma borda:
# nenhuma classe mistura local onde Lula venceu com local onde Flávio venceu.
BORDAS_A = [0, 30, 40, 50, 60, 70, 100]          # 6 classes, pontas abertas
BORDAS_B = [10, 20, 30, 40, 50, 60, 70, 80, 90]  # 8 classes de 10 pontos


def raio(total, total_max, raio_max=20):
    """Raio em pixels com ÁREA proporcional aos votos (por isso a raiz quadrada).
    Se o raio fosse proporcional aos votos, um local com o dobro de votos teria 4x a área
    e pareceria muito maior do que é."""
    return 3 + (raio_max - 3) * np.sqrt(total / total_max)


def pct_br(valor):
    """57.6 -> '57,6%' (vírgula decimal, padrão brasileiro)."""
    return f"{valor:.1f}%".replace(".", ",")


def num_br(valor):
    """2074 -> '2.074' (ponto como separador de milhar, padrão brasileiro)."""
    return f"{valor:,}".replace(",", ".")


def cartao_html(local):
    """Monta o HTML do tooltip de um ponto do mapa."""
    if local["LULA"] > local["FLAVIO"]:
        lider = f'<span style="color:{VERMELHO_LULA}">Lula</span> lidera com {pct_br(local["PCT_LULA"])}'
    elif local["FLAVIO"] > local["LULA"]:
        lider = f'<span style="color:{AZUL_FLAVIO}">Flávio</span> lidera com {pct_br(local["PCT_FLAVIO"])}'
    else:
        lider = "Empate"

    bairro = f'<div style="color:#666; font-size:12px; margin-bottom:8px;">Bairro: {local["NM_BAIRRO"]}</div>' if local["NM_BAIRRO"] else ""
    return f"""
    <div style="min-width:240px; max-width:320px; white-space:normal;">
      <div style="font-weight:700; font-size:14px; margin-bottom:2px;">{local["NM_LOCAL_VOTACAO"]}</div>
      <div style="color:#666; font-size:12px; margin-bottom:2px;">Zona 328 · Seções {local["SECOES"]}</div>
      {bairro}
      <div style="font-weight:700; margin-bottom:4px;">{lider}</div>
      <div style="display:flex; height:10px; border-radius:5px; overflow:hidden; margin-bottom:4px;">
        <div style="width:{local["PCT_LULA"]:.1f}%; background:{VERMELHO_LULA};"></div>
        <div style="width:{local["PCT_FLAVIO"]:.1f}%; background:{AZUL_FLAVIO};"></div>
      </div>
      <div style="display:flex; justify-content:space-between; font-size:12px;">
        <span style="color:{VERMELHO_LULA}">Lula {pct_br(local["PCT_LULA"])}</span>
        <span style="color:{AZUL_FLAVIO}">Flávio {pct_br(local["PCT_FLAVIO"])}</span>
      </div>
      <div style="color:#666; font-size:12px; margin-top:6px;">{num_br(local["TOTAL"])} votos ({num_br(local["BRANCO"] + local["NULO"])} brancos/nulos)</div>
    </div>"""


# Opacidade do mapa de fundo (0 = invisível, 1 = normal). Ajuste olhando o resultado.
OPACIDADE_FUNDO = 0.6


# Estilo da "caixa" do tooltip (CSS). Fundo claro para as cores dos partidos terem contraste.
ESTILO_TOOLTIP = """
    background-color: #ffffff;
    color: #222222;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 13px;
    border-radius: 8px;
    border: 1px solid #cccccc;
    padding: 10px 12px;
    box-shadow: 3px 3px 10px rgba(0, 0, 0, 0.25);
"""


def rotulos_classes(bordas):
    """[0, 30, 40, ..., 100] -> ['até 30', '30–40', ..., '70 ou mais']"""
    rotulos = []
    for i, (inicio, fim) in enumerate(zip(bordas[:-1], bordas[1:])):
        if i == 0 and inicio == 0:
            rotulos.append(f"até {fim}")
        elif i == len(bordas) - 2 and fim == 100:
            rotulos.append(f"{inicio} ou mais")
        else:
            rotulos.append(f"{inicio}–{fim}")
    return rotulos


def legenda_html(bordas, cores, titulo, cor_empate=None):
    """Legenda com todas as classes do mesmo tamanho (a legenda padrão do branca é proporcional ao valor)."""
    caixas = "".join(
        f'''<div style="flex:1; text-align:center;">
             <div style="height:14px; background:{cor}; border:1px solid #999;"></div>
             <div style="font-size:11px; margin-top:2px;">{rotulo}</div>
           </div>'''
        for cor, rotulo in zip(cores, rotulos_classes(bordas))
    )
    caixa_empate = ""
    if cor_empate:
        caixa_empate = f'''
      <div style="display:flex; align-items:center; gap:6px; font-size:11px; margin-top:6px;">
        <div style="width:14px; height:14px; background:{cor_empate}; border:1px solid #999;"></div>
        Empate exato (mesmo número de votos)
      </div>'''
    return f'''
    <div style="position:fixed; top:12px; right:12px; z-index:9999; width:340px;
                background:rgba(255,255,255,0.92); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,0.3); font-family:Arial, sans-serif; color:#222;">
      <div style="font-weight:700; font-size:12px; margin-bottom:6px;">{titulo}</div>
      <div style="display:flex; gap:2px;">{caixas}</div>{caixa_empate}
      <div style="display:flex; justify-content:space-between; font-size:11px; margin-top:4px;">
        <span style="color:{AZUL_FLAVIO};">&#9664; Flávio venceu</span>
        <span style="color:{VERMELHO_LULA};">Lula venceu &#9654;</span>
      </div>
    </div>'''


def mapa_2turno(dados, bordas, cores, titulo, cor_empate="#ffffff"):
    escala = cm.StepColormap(
        colors=cores,
        index=bordas,
        vmin=bordas[0],
        vmax=bordas[-1],
    )   # usado só para calcular a cor de cada círculo; a legenda é a legenda_html()

    # Prepara as colunas que vão para o GeoJSON
    pontos = dados.copy()
    pontos["RAIO"] = raio(pontos["TOTAL"], pontos["TOTAL"].max())
    # Limita o valor ao intervalo da escala (na opção B, 9,6% vira 10%)
    pontos["PCT_COR"] = pontos["PCT_LULA"].clip(bordas[0], bordas[-1] - 1e-9)
    # Cor de cada círculo. Empate exato (50%) cairia na classe 50–60 e pareceria vitória do Lula,
    # então recebe uma cor neutra própria: branco, o centro da paleta divergente.
    # (Testamos cinza, mas ele se confundia com o mapa de fundo, que também é cinza.)
    empate = pontos["LULA"] == pontos["FLAVIO"]
    pontos["COR"] = pontos["PCT_COR"].apply(escala)
    pontos.loc[empate, "COR"] = cor_empate
    pontos["CARTAO"] = pontos.apply(cartao_html, axis=1)
    # Desenha os maiores primeiro, para os pequenos ficarem por cima e continuarem acessíveis
    pontos = pontos.sort_values("TOTAL", ascending=False)

    # Tabela -> GeoDataFrame de pontos (longitude = x, latitude = y; EPSG:4326 = lat/long comum)
    gdf = gpd.GeoDataFrame(
        pontos[["CARTAO", "RAIO", "COR"]],
        geometry=gpd.points_from_xy(pontos["NR_LONGITUDE"], pontos["NR_LATITUDE"]),
        crs="EPSG:4326",
    )

    # Mapa base: mesmo servidor do notebook da aula (funciona sem chave de API).
    # O CartoDB Positron, recomendado no slide 7 para dados estatísticos, hoje exige chave.
    # Para ter um efeito parecido, deixamos o fundo em escala de cinza (filtro CSS) e mais claro (opacidade):
    # assim a única informação colorida do mapa é o dado eleitoral.
    m = folium.Map(tiles=None)
    folium.TileLayer(
        tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
        attr="&copy; OpenStreetMap contributors",
        name="Mapa base (OpenStreetMap em cinza)",
        opacity=OPACIDADE_FUNDO,
        class_name="fundo-cinza",     # "etiqueta" usada pelo CSS abaixo
    ).add_to(m)
    m.get_root().header.add_child(folium.Element(
        "<style>"
        ".fundo-cinza { filter: grayscale(100%); }"
        ".leaflet-container { background: #ffffff; }"   # fundo branco atrás dos tiles (o padrão é cinza)
        "</style>"
    ))

    folium.GeoJson(
        gdf,
        name="2º turno",
        marker=folium.CircleMarker(),   # cada ponto do GeoJSON vira um círculo
        style_function=lambda f: {
            "radius": f["properties"]["RAIO"],
            "fill": True,               # o CircleMarker vem sem preenchimento por padrão
            "fillColor": f["properties"]["COR"],
            "fillOpacity": 0.85,
            "color": "#555555",         # borda cinza: deixa visíveis os círculos de cor clara (perto de 50%)
            "weight": 0.8,
        },
        highlight_function=lambda f: {"weight": 3, "color": "#000000"},   # destaca o círculo sob o mouse
        tooltip=folium.GeoJsonTooltip(
            fields=["CARTAO"],
            labels=False,               # sem o rótulo "CARTAO:", só o conteúdo
            sticky=True,
            style=ESTILO_TOOLTIP,
        ),
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(legenda_html(bordas, cores, titulo, cor_empate if empate.any() else None)))   # legenda
    # Enquadra o mapa em todos os pontos: total_bounds = [oeste, sul, leste, norte]
    oeste, sul, leste, norte = gdf.total_bounds
    m.fit_bounds([[sul, oeste], [norte, leste]])
    return m

In [50]:
# Quantos locais caem em cada classe? Ajuda a comparar as duas opções
print("Opção A:")
print(pd.cut(por_local["PCT_LULA"], BORDAS_A).value_counts().sort_index())
print()
print("Opção B:")
print(pd.cut(por_local["PCT_LULA"].clip(10, 90), BORDAS_B, include_lowest=True).value_counts().sort_index())

Opção A:
PCT_LULA
(0, 30]       2
(30, 40]      3
(40, 50]     31
(50, 60]     15
(60, 70]      9
(70, 100]     2
Name: count, dtype: int64

Opção B:
PCT_LULA
(9.999, 20.0]     1
(20.0, 30.0]      1
(30.0, 40.0]      3
(40.0, 50.0]     31
(50.0, 60.0]     15
(60.0, 70.0]      9
(70.0, 80.0]      2
(80.0, 90.0]      0
Name: count, dtype: int64


In [51]:
from pathlib import Path
Path("mapas").mkdir(exist_ok=True)

mapa_2t = mapa_2turno(por_local, BORDAS_A, RDBU_6, "2º turno – % dos votos válidos em Lula")
mapa_2t.save("mapas/mapa_2turno.html")
mapa_2t

#### Relatório – Mapa do 2º turno

**O fato**
- **Empate técnico:** Flávio 50,1% × Lula 49,9% dos votos válidos, diferença de 133 votos (brancos e nulos: 3,6%).
- **Flávio venceu em mais locais (34 × 26, e 2 empates), mas Lula venceu em locais menores** (mediana de 572 votos, contra 892).
- **Na cidade, o padrão é leste × oeste do centro histórico:** a leste (Matosinhos, Centro, Fábricas, Pio XII), Flávio teve 52,3%; a oeste (Tejuco, Senhor dos Montes, Bonfim, São Geraldo), Lula teve 56,6%.
- **Sede e distritos têm resultado parecido** (Lula 50,2% × 48,5%), **mas os extremos estão na zona rural:** Capela do Zueira (9,6% Lula) e Canela (73,1%).
- **UFSJ:** os campi do centro votaram mais em Lula (Dom Bosco 60,3%, Santo Antônio 53,3%); o CTAN, fora do centro, em Flávio (54,8%).
- *Ressalva:* o mapa mostra onde as pessoas votam, não onde moram.

**O design**
- **Paleta divergente RdBu** (vermelho = Lula/PT, azul = Flávio/PL), com o centro neutro fixado em 50% (empate). A escala automática colocaria o centro em 47,4%.
- **6 classes** simétricas (até 30 | 30–40 | 40–50 ‖ 50–60 | 60–70 | 70+): 86% dos votos estão entre 40% e 60%; a opção com 8 classes deixava uma classe vazia.
- **Empates exatos em branco**, para não parecerem vitória de um dos lados.
- **Círculos** (um por prédio) com **área proporcional ao total de votos**; não temos polígonos das seções.
- **Mapa base** OpenStreetMap em cinza e com opacidade 0,6: a única informação colorida é o resultado (o CartoDB Positron exige chave de API).
- **Folium:** `GeoJson` + `CircleMarker`, `GeoJsonTooltip` com cartão em HTML/CSS e legenda própria em HTML.